# Vector Store Content Ingestion
We are going to load content about Paestum fromvarious sources and in different formats into vector store for later RAG chatbot developement.

### References
This notebook is reference from manning book **AI Agents and Applications - CH07** 

## Loaders

In [1]:
from langchain_community.document_loaders import (
    WikipediaLoader,
    Docx2txtLoader,
    PyPDFLoader,
    TextLoader
)

## Vector Store
We need an Embedding model to create an instance of Vector-store.

In [6]:
from langchain_google_vertexai import VertexAIEmbeddings

embeddings_model = VertexAIEmbeddings(model="gemini-embedding-001")

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_8492\105074813.py:3: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings_model = VertexAIEmbeddings(model="gemini-embedding-001")


In [7]:
#from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma

persist_directory = "./chroma_tourist_info_db"
vector_db = Chroma("tourist_info", embeddings_model, persist_directory=persist_directory)

## Load contents to Vector Store from different sources with different formats

In [10]:
### we need a splitter to split content to chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)

In [11]:
wikipedia_loader = WikipediaLoader(query="Paestum")
wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
vector_db.add_documents(wikipedia_chunks)

['416352f6-0b58-4c8e-9888-48fce22d309b',
 '6b293ee0-db3b-47a1-83e1-6a0cfb7dc5a3',
 '31ba9e3f-90dc-444b-b429-57f7c79c5194',
 '35759656-42c1-4da9-843e-6935e7858996',
 '7057e99e-ec1a-498d-bbde-5eecacbe3b13',
 '4e3f9ebd-5767-4331-b996-a2fde26d8e61',
 '5929da93-d980-42e2-9308-fbb1612c5c3e',
 '29f8a79e-d7cf-4070-8098-1ef1e1d04c68',
 '4af980be-e29c-40ea-9ba8-a02d3829d7f4',
 '51b09b9c-cf50-4ef2-b2ef-7372f7695499',
 '8e92c4b8-94f9-4a81-ba0a-243a9aab415e',
 '16281320-b1ed-4466-a043-c4b26600c9bf',
 'ae2ce0af-2eac-4b82-90f7-0bac66323ced',
 'dc3bde69-bf9c-4520-af7f-da5a0691f45f',
 'a6c243ce-b17f-4fd1-bbe6-a42197e22f86',
 '95658d14-38f2-4b76-b494-b39415f722a6',
 '49231eff-df3c-49bd-9533-05e11cf8418a',
 'd896660a-d50c-4532-81da-b0bad4ddd9bc',
 '818026fa-b9ef-4291-b13c-07e50cc943df',
 'ce980488-7372-49f0-bab5-94b72fd27024',
 '99842d20-ea91-4b77-b760-04979e62ae10',
 '0ca5680d-30e9-49e7-aed8-6ed91c7e844b',
 'f48aef45-d912-47b3-a8cd-a3ddd7e29095',
 'cb40e555-8f57-4ba4-a59a-2f28627f184e',
 '2e6c8d0f-1227-

we can have reusable function to insert content with different sources & formats

In [13]:
def split_and_import(vector_db, loader, splitter):
    chunks = splitter.split_documents(loader.load())
    _ = vector_db.add_documents(chunks)
    return vector_db

load contents from `docx` file

In [20]:
word_loader = Docx2txtLoader("paestum_docs/Paestum-Britannica.docx")
vector_db = split_and_import(vector_db, word_loader, text_splitter)

In [21]:
pdf_loader = PyPDFLoader("paestum_docs/PaestumRevisited.pdf")
vector_db = split_and_import(vector_db, pdf_loader, text_splitter)

## Query the Vector Store

In [8]:
from rich import print as pprint

In [9]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4)
pprint(results)

[
    Document(
        id='4e3f9ebd-5767-4331-b996-a2fde26d8e61',
        metadata={
            'summary': 'Paestum ( PEST-əm, US also  PEE-stəm, Latin: [ˈpae̯stũː]) was a major ancient Greek city on
the coast of the Tyrrhenian Sea, in Magna Graecia. The ruins of Paestum are famous for their three ancient Greek 
temples in the Doric order dating from about 550 to 450 BCE that are in an excellent state of preservation. The 
city walls and amphitheatre are largely intact, and the bottom of the walls of many other structures remain, as 
well as paved roads. The site is open to the public, and there is a modern national museum within it, which also 
contains the finds from the associated Greek site of Foce del Sele.\nPaestum was established around 600 BCE by 
settlers from Sybaris, a Greek colony in southern Italy, under the name of Poseidonia (Ancient Greek: Ποσειδωνία). 
The city thrived as a Greek settlement for about two centuries, witnessing the development of democracy. In 400 
BCE, the Lucanians seized the city. Romans took over in 273 BCE, renaming it Paestum and establishing a Latin 
colony. Later, its decline ensued from shifts in trade routes and the onset of flooding and marsh formation. As 
Pesto or Paestum, the town became a bishopric (now only titular), but it was abandoned in the Early Middle Ages, 
and left undisturbed and largely forgotten until the eighteenth century.\nToday the remains of the city are found 
in the modern frazione of Paestum, which is part of the comune of Capaccio Paestum in the Province of Salerno in 
the region of Campania, Italy. The modern settlement, directly to the south of the archaeological site, is a 
popular seaside resort with long sandy beaches. The Paestum railway station on the Naples-Salerno-Reggio Calabria 
railway line is directly to the east of the ancient city walls.',
            'source': 'https://en.wikipedia.org/wiki/Paestum',
            'title': 'Paestum'
        },
        page_content='== Name ==\nThe Greek settlers who founded the city originally named it Poseidonia (Ancient 
Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed 
it to Paistos and the Romans gave the city its current name.\n\n\n== Ancient ruins and features =='
    ),
    Document(
        id='03c950e8-375c-4943-acbf-d6b569336c6e',
        metadata={'source': 'paestum_docs/Paestum-Britannica.docx'},
        page_content='Poseidonia was probably founded about 600\xa0BC\xa0by Greek colonists from\xa0Sybaris, along 
the\xa0Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ 
resistance the city came under the domination of the\xa0Lucanians\xa0(an\xa0indigenous\xa0Italic people) sometime 
before 400\xa0BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the 
Lucanians at Paestum about 332\xa0BC, but the city remained Lucanian until 273, when it came under'
    ),
    Document(
        id='31ba9e3f-90dc-444b-b429-57f7c79c5194',
        metadata={
            'summary': 'Paestum ( PEST-əm, US also  PEE-stəm, Latin: [ˈpae̯stũː]) was a major ancient Greek city on
the coast of the Tyrrhenian Sea, in Magna Graecia. The ruins of Paestum are famous for their three ancient Greek 
temples in the Doric order dating from about 550 to 450 BCE that are in an excellent state of preservation. The 
city walls and amphitheatre are largely intact, and the bottom of the walls of many other structures remain, as 
well as paved roads. The site is open to the public, and there is a modern national museum within it, which also 
contains the finds from the associated Greek site of Foce del Sele.\nPaestum was established around 600 BCE by 
settlers from Sybaris, a Greek colony in southern Italy, under the name of Poseidonia (Ancient Greek: Ποσειδωνία). 
The city thrived as a Greek settlement for about two centuries, witnessing the development of democracy. In 400 
BCE, the Luca

# Build RAG chatbot chain

## Build the basic chain
We need :
1. Retriever
2. Prompt
3. RunnablePassThrough

In [10]:
retriever = vector_db.as_retriever()

In [11]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """
Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.
{context}
Question: {question}
Helpful Answer:
"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

In [12]:
from langchain_core.runnables import RunnablePassthrough

question_feeder = RunnablePassthrough()

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatbot = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

In [14]:
rag_chain = {'context':retriever, 'question':question_feeder}|rag_prompt|chatbot

In [15]:
question = "Where was Poseidonia and who renamed it to Paestum. Also tell me the source."
answer = rag_chain.invoke(question)
pprint(answer)

AIMessage(
    content='Poseidonia was a major ancient Greek city located on the coast of the Tyrrhenian Sea in Magna Graecia.
The Lucanians renamed it Paistos, and the Romans later gave it the name Paestum. The source for this information is
Wikipedia.',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash-lite',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019d1379-3164-7c00-8aee-87a79547b0bd-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1695,
        'output_tokens': 53,
        'total_tokens': 1748,
        'input_token_details': {'cache_read': 0}
    }
)

In [16]:
print(answer.content)

Poseidonia was a major ancient Greek city located on the coast of the Tyrrhenian Sea in Magna Graecia. The Lucanians renamed it Paistos, and the Romans later gave it the name Paestum. The source for this information is Wikipedia.


****
In the current setup, the chatbot chain does not have memory, that means it could not deal with follow up question well.

In [17]:
### the follow up question
question = 'And then, what they do? Tell me only if you know. Also tell me the source'
answer = rag_chain.invoke(question)
pprint(answer)

AIMessage(
    content='I don\'t know what "they" do. The provided context does not contain information about what actions 
were taken.',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash-lite',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019d137f-0ae0-7ce0-bd07-243d649733ea-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1063,
        'output_tokens': 24,
        'total_tokens': 1087,
        'input_token_details': {'cache_read': 0}
    }
)

## Adding conversation memory to the RAG chain
Most chat-oriented LLMs use a standard format for messages which associates roles with chat messages. A chat message is typically a key-value pair like `("role": "message text")`.<br>
**Memory-enabled** RAG design requires message history which fulfills chat standard format.

### Chat-based Prompt

In [50]:
from langchain_core.prompts import ChatPromptTemplate

rag_role_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

In [51]:
from langchain_community.chat_message_histories import ChatMessageHistory

chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

In [52]:
retriever = vector_db.as_retriever()

from langchain_google_genai import ChatGoogleGenerativeAI

chatbot = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

from langchain_core.runnables import RunnablePassthrough

question_feeder = RunnablePassthrough()

In [53]:
from langchain_core.runnables import RunnableLambda

rag_memory_chain = {
    "retrieved_context": retriever, 
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_role_prompt | chatbot

In [54]:
def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')                                      
    return answer

In [55]:
question = 'Where was Poseidonia and who renamed it to Paestum. Also tell me the source.'
answer = execute_chain_with_memory(rag_memory_chain, question)
pprint(answer)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed it to Paestum. Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was a major ancient Greek city on the coast of the Tyrrhenian Sea in Magna Graecia. The Lucanians renamed it Paistos, and the Romans later gave the city its current name, Paestum. The source for this information is Wikipedia.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d15c2-323d-72e0-96c9-7d0931183806-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1734, 'output_tokens': 55, 'total_tokens': 1789, 'input_token_details': {'cache_read': 0}})]




AIMessage(
    content='Poseidonia was a major ancient Greek city on the coast of the Tyrrhenian Sea in Magna Graecia. The 
Lucanians renamed it Paistos, and the Romans later gave the city its current name, Paestum. The source for this 
information is Wikipedia.',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash-lite',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019d15c2-323d-72e0-96c9-7d0931183806-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1734,
        'output_tokens': 55,
        'total_tokens': 1789,
        'input_token_details': {'cache_read': 0}
    }
)

***
Let's try follow up question

In [56]:
### the follow up question
question = 'And then, what they do? Tell me only if you know. Also tell me the source'
answer = execute_chain_with_memory(rag_memory_chain, question)
pprint(answer)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed it to Paestum. Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was a major ancient Greek city on the coast of the Tyrrhenian Sea in Magna Graecia. The Lucanians renamed it Paistos, and the Romans later gave the city its current name, Paestum. The source for this information is Wikipedia.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d15c2-323d-72e0-96c9-7d0931183806-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1734, 'output_tokens': 55, 'total_tokens': 1789, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='And then, what they do? Tell me only if you know. Also tell me the source', additional_kwargs={}, response_metadata={}), AIMessage(content='The people of Poseidonia, la

AIMessage(
    content='The people of Poseidonia, later Paestum, built impressive Doric temples, including the First Temple of
Hera and the Second Temple of Hera. They also created unique funerary art, such as the Tomb of the Diver, which 
features a striking fresco of a lone diver. The sources for this information are Wikipedia articles on these 
respective sites.',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash-lite',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019d15c2-41b3-78a1-a061-8958644a609c-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1177,
        'output_tokens': 70,
        'total_tokens': 1247,
        'input_token_details': {'cache_read': 0}
    }
)

In [58]:
pprint(chat_history_memory.messages)

[
    HumanMessage(
        content='Where was Poseidonia and who renamed it to Paestum. Also tell me the source.',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='Poseidonia was a major ancient Greek city on the coast of the Tyrrhenian Sea in Magna Graecia. The
Lucanians renamed it Paistos, and the Romans later gave the city its current name, Paestum. The source for this 
information is Wikipedia.',
        additional_kwargs={},
        response_metadata={
            'finish_reason': 'STOP',
            'model_name': 'gemini-2.5-flash-lite',
            'safety_ratings': [],
            'model_provider': 'google_genai'
        },
        id='lc_run--019d15c2-323d-72e0-96c9-7d0931183806-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 1734,
            'output_tokens': 55,
            'total_tokens': 1789,
            'input_token_details': {'cache_read': 0}
        }
    ),
    HumanMessage(
        content='And then, what they do? Tell me only if you know. Also tell me the source',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='The people of Poseidonia, later Paestum, built impressive Doric temples, including the First 
Temple of Hera and the Second Temple of Hera. They also created unique funerary art, such as the Tomb of the Diver,
which features a striking fresco of a lone diver. The sources for this information are Wikipedia articles on these 
respective sites.',
        additional_kwargs={},
        response_metadata={
            'finish_reason': 'STOP',
            'model_name': 'gemini-2.5-flash-lite',
            'safety_ratings': [],
            'model_provider': 'google_genai'
        },
        id='lc_run--019d15c2-41b3-78a1-a061-8958644a609c-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 1177,
            'output_tokens': 70,
            'total_tokens': 1247,
            'input_token_details': {'cache_read': 0}
        }
    )
]